# Chapter 1 — Introduction
### Notebook 5 · Agentic lab — build, formalise, evaluate, optimise, improve

*Book reference: Extends Ch. 1 beyond the book*

Notebooks 1–4 built a *program* that triages an ontology. This notebook builds an **agent** that does the same job, and then does the harder part: says how good it is, proves the number, and improves it without fooling itself.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
sys.path.insert(0, str(Path.cwd()))          # so ch01_toolkit imports
import ch01_toolkit as ch1
from oe_course import ontology as ont
from oe_course.data import corpus
import pandas as pd
pd.set_option("display.width", 120)

In [3]:
import json
from oe_course import (agents, evaluation as ev, llm, mdp,
                       optimize as opt, programs as pr, tools as T)
from oe_course.config import offline
print('offline mode (simulated LLM):', offline())

offline mode (simulated LLM): True


**By the end of this notebook you can:**

1. Expose domain capabilities as **function tools** with trigger conditions, and log every call.
2. Compare a single agent loop against an explicitly **decomposed** pipeline on score *and* cost.
3. Formalise the task as an **MDP**, solve it exactly, and report the agent's **regret** against the optimal policy.
4. Build an evaluation dataset and three kinds of metric: deterministic, LLM-as-judge, and **GEPA-shaped feedback**.
5. Optimise the agent's instruction with **DSPy GEPA** and attribute the gain to a specific instruction change.
6. Package the result as a versioned **skill**, and run a **self-improvement** loop behind a held-out promotion gate.

> **Read the mode line above.** Offline, the LLM is a deterministic simulator: a weak agent that follows explicit instructions and ignores everything else. It makes every number here reproducible and every optimisation real — but the numbers describe *the simulator*, not Claude. Set `ANTHROPIC_API_KEY` to run the identical code against a live model and get numbers about the model.

## 1. Function tools

An agent can only do what its tools let it do, so tool design *is* agent design. Three conventions are used throughout this course, and each one is load-bearing:

1. **Tools are bound to a workspace, not to globals.** A `ToolContext` holds the store and the artefact; tools close over it. Two students, two contexts, no shared state.
2. **Descriptions state a trigger, not just a behaviour.** *"Call this whenever asked to review or critique an ontology"* beats *"scans for defects"*. Models select tools from the description; a description that omits *when* leaves the choice to chance.
3. **Every call is logged.** The log is the trajectory, the cost, and — in §3 — the MDP episode.

In [4]:
ctx = T.ToolContext()
toolset = T.build_toolset(ctx)
for t in toolset:
    params = list(t.args_schema.model_json_schema().get('properties', {}))
    print(f'{t.name:20s} {params}')
    print(f'{"":20s} {t.description.splitlines()[0]}')

load_artefact        ['name']
                     Load a named ontology from the course corpus into the workspace.
list_artefacts       []
                     List the ontology names available in the course corpus.
graph_metrics        []
                     Count the loaded ontology's classes, properties, axioms and annotations.
spectrum_position    []
                     Classify the loaded ontology on the ontology spectrum, with evidence.
scan_smells          []
                     Detect known modelling defects in the loaded ontology.
smell_catalogue      []
                     Explain every defect the scanner can detect and why each one matters.
sparql_select        ['query']
                     Run a SPARQL SELECT against the loaded ontology and return the rows.
sparql_ask           ['query']
                     Run a SPARQL ASK against the loaded ontology; returns true or false.


### The SPARQL tool talks to Fuseki when Fuseki is there

`SparqlStore.auto()` probes the local Fuseki (`infra/fuseki`, `docker compose up -d`) and silently falls back to an in-memory rdflib dataset with the same API. The agent code does not change — which is the point: the **tool boundary** is what makes the backend swappable.

In [5]:
from oe_course.sparql import SparqlStore, fuseki_available
print('Fuseki reachable:', fuseki_available())
auto = SparqlStore.auto()
print('agent will use backend:', auto.backend)

auto.load_text(corpus.get('awo').turtle)
rows = auto.select('SELECT ?c WHERE { ?c rdfs:subClassOf awo:Herbivore }')
print('herbivores:', [r['c'].split('#')[-1] for r in rows])

Fuseki reachable: False
agent will use backend: rdflib-memory


herbivores: ['Giraffe', 'Impala', 'RockDassie']


## 2. One loop, or a decomposed pipeline?

The obvious construction is a single ReAct-style loop: one model, one tool belt, one prompt, model decides the order. Let's run it and read the trajectory.

In [6]:
agent, actx = agents.build_agent()
run = agents.run_agent(agent, actx, "Triage the artefact named 'legacy-import'.")
print('trajectory:', ' -> '.join(actx.log.names()))
print('cost:', actx.log.summary())
print('\nanswer:\n', run.answer)

C:\Users\marci\anaconda3\envs\DATAENLIGHT_AI_LAB\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


trajectory: load_artefact -> graph_metrics -> spectrum_position -> scan_smells
cost: {'n_calls': 4, 'distinct_tools': ['graph_metrics', 'load_artefact', 'scan_smells', 'spectrum_position'], 'failures': 0, 'seconds': 0.004}

answer:
 {
 "level": "taxonomy",
 "smells": [
  "missing-label",
  "property-without-domain-or-range",
  "subsumption-cycle"
 ],
 "evidence": [
  "3 classes, 3 labels -> named terms exist",
  "2 subsumption axioms -> a hierarchy exists"
 ]
}


Now the **functionally decomposed** alternative: `plan → act → draft → critique` as separate LangGraph nodes. It costs more to build and constrains the model. It buys three things:

* each stage can be prompted, evaluated and optimised **independently**;
* the `critique` stage is a *verification* step with access to the raw tool output, so a hallucinated defect id **cannot survive it**;
* the trajectory is legible by construction.

In [7]:
pipe, pctx = agents.build_pipeline()
out = pipe.invoke({'task': 'triage', 'artefact': 'legacy-import'})
print('answer :', json.dumps(out['answer'], indent=1))
print('critique:', out['critique'])
print('cost    :', pctx.log.summary())

answer : {
 "level": "taxonomy",
 "smells": [
  "missing-label",
  "property-without-domain-or-range",
  "subsumption-cycle"
 ],
 "justification": "3 classes, 3 labels -> named terms exist; 2 subsumption axioms -> a hierarchy exists"
}
critique: all claims supported
cost    : {'n_calls': 4, 'distinct_tools': ['graph_metrics', 'load_artefact', 'scan_smells', 'spectrum_position'], 'failures': 0, 'seconds': 0.004}


### The critic earns its keep only when the drafter is wrong

Above, the critic reported *"all claims supported"* — it did nothing. That is the honest result, and it is why decomposition should be **measured, not assumed**. Let's inject a hallucinated defect and watch the critic catch what a single loop would have emitted.

In [8]:
evidence = {'scan_smells': [{'smell': 'subsumption-cycle', 'subject': 'ex:Process'}]}
draft = {'level': 'taxonomy',
         'smells': ['subsumption-cycle', 'class-as-individual'],  # 2nd is invented
         'justification': '...'}
found = {f['smell'] for f in evidence['scan_smells']}
kept = sorted(set(draft['smells']) & found)
dropped = sorted(set(draft['smells']) - found)
print('claimed  :', draft['smells'])
print('supported:', kept)
print('DROPPED as unsupported:', dropped)
print('\nA verification stage with access to raw tool output turns a whole class\n'
      'of hallucination into an impossibility rather than an unlikelihood.')

claimed  : ['subsumption-cycle', 'class-as-individual']
supported: ['subsumption-cycle']
DROPPED as unsupported: ['class-as-individual']

A verification stage with access to raw tool output turns a whole class
of hallucination into an impossibility rather than an unlikelihood.


## 3. The task as a Markov decision process

"Is the agent efficient?" is unanswerable as posed. Formalise the task as an MDP and it becomes arithmetic.

| | Triage task |
|---|---|
| **S** | the evidence gathered so far, plus whether we have committed |
| **A** | one tool call per kind of evidence, plus `submit` |
| **T** | deterministic for a fixed artefact — the tools are pure functions of the graph |
| **R** | `-cost` per tool call; on `submit`, the **evaluation metric's score** |
| **γ** | 1.0 — finite horizon, no reason to discount |

The reward deliberately *contains* the evaluation metric. The thing GEPA optimises, the thing the grader measures, and the thing the MDP rewards must be one function; if they differ, you are optimising something you are not measuring.

In [9]:
EVIDENCE = ['metrics', 'spectrum', 'smells', 'catalogue', 'sparql']

def answer_quality(evidence):
    """What score is achievable from this evidence set?

    Spectrum and smells each carry half the marks (mirroring triage_scorer).
    The other tools are legitimate but do not add score -- they are the
    temptation the cost term exists to resist.
    """
    return (0.5 * ('spectrum' in evidence)) + (0.5 * ('smells' in evidence))

M = mdp.EvidenceMDP(EVIDENCE, answer_quality, step_cost=0.05)
V, pi = mdp.value_iteration(M)
s0 = M.initial_state()
print(f'|S| = {len(M.states())}, V*(s0) = {V[s0]:.3f}')
best = mdp.run_episode(M, mdp.greedy_policy(pi))
print('optimal policy:', ' -> '.join(best.actions))
print('optimal return:', round(best.discounted_return(), 3))

|S| = 64, V*(s0) = 0.900
optimal policy: spectrum -> smells -> submit
optimal return: 0.9


The optimal policy buys **exactly** the two evidence kinds that carry score and then commits. Now replay what our real agent did, in the same vocabulary, and measure the gap.

In [10]:
TOOL_TO_EVIDENCE = {
    'graph_metrics': 'metrics', 'spectrum_position': 'spectrum',
    'scan_smells': 'smells', 'smell_catalogue': 'catalogue',
    'sparql_select': 'sparql', 'sparql_ask': 'sparql',
    # load_artefact / list_artefacts map to nothing: overhead, but still charged
}
gold = {a.name: a for a in corpus.CORPUS}['legacy-import']
agent_answer = json.loads(run.answer)
achieved = ev.set_f1(agent_answer['smells'], gold.gold_smells)[2] * 0.5 \
         + ev.exact_match(agent_answer['level'], gold.gold_level) * 0.5

episode = mdp.episode_from_tool_log(actx.log, M, TOOL_TO_EVIDENCE, achieved)
for t in episode.transitions:
    print(f'  {str(t.state):28s} --{t.action:18s}--> r={t.reward:+.2f}')
G = episode.discounted_return()
print(f'\nagent return   G = {G:.3f}')
print(f'optimal        V* = {V[s0]:.3f}')
print(f'REGRET            = {V[s0] - G:.3f}')

  [-]                          --load_artefact     --> r=-0.05
  [-]                          --graph_metrics     --> r=-0.05
  [metrics]                    --spectrum_position --> r=-0.05
  [metrics,spectrum]           --scan_smells       --> r=-0.05
  [metrics,smells,spectrum]    --submit            --> r=+1.00

agent return   G = 0.800
optimal        V* = 0.900
REGRET            = 0.100


> **What the regret is telling you.** The agent scored full marks on the answer, so all of its regret is *procedural*: it paid for `load_artefact` (unavoidable overhead this model does not credit) and for `graph_metrics` (genuinely unnecessary for the score as defined). This is a far more actionable diagnosis than "the agent seems a bit chatty" — and notice it is a criticism of the **reward model** as much as of the agent: if you believe metrics *should* be gathered, the reward is wrong, not the agent.

In [11]:
print('random policy value :', round(mdp.policy_value(M, mdp.random_policy(), 400), 3))
print('agent return        :', round(G, 3))
print('optimal value       :', round(V[s0], 3))
print('\nAn agent that beats random but trails optimal is the normal case.\n'
      'The number to report is the gap, and where it comes from.')

random policy value : 0.36
agent return        : 0.8
optimal value       : 0.9

An agent that beats random but trails optimal is the normal case.
The number to report is the gap, and where it comes from.


## 4. Evaluation: a dataset and three kinds of metric

### 4.1 The dataset
Eleven artefacts with **hand-written** gold labels, split **by artefact** — never by random row. Leaking an artefact across the split would let the optimiser memorise its answer and report a number that means nothing.

In [12]:
train = ev.build_triage_dataset('train')
dev = ev.build_triage_dataset('dev')
print(f'train {len(train)}, dev {len(dev)} (disjoint artefacts)')
assert not ({e.artefact for e in train} & {e.artefact for e in dev})
for e in dev:
    print(f'  {e.artefact:26s} {e.level:22s} {e.smells}')

train 6, dev 5 (disjoint artefacts)
  staff-instance-confusion   taxonomy               ['individual-as-class']
  loose-ends                 taxonomy               ['undeclared-term']
  opaque-ids                 taxonomy               ['missing-label', 'no-disjointness']
  bare-properties            taxonomy               ['no-disjointness', 'property-without-domain-or-range']
  legacy-import              taxonomy               ['missing-label', 'property-without-domain-or-range', 'subsumption-cycle']


### 4.2 Deterministic metric
Half the mark for the spectrum level, half for the F1 over defect ids. Cheap, reproducible, and the only sound basis for a regression test. It cannot judge prose — that is the next metric's job.

In [13]:
class FakePred:
    level, smells = 'taxonomy', ['subsumption-cycle', 'missing-label']
gold_ex = [e for e in train if e.artefact == 'broken-cycle'][0]
report = ev.triage_scorer(gold_ex, FakePred())
print('score:', round(report.score, 3))
for n in report.notes:
    print('  note:', n)
print('  violated rules:', report.violated)

score: 0.75
  note: Failed to report these real defects: ['no-disjointness'].
  note: Reported defects that the scanner did not find: ['missing-label'].
  note: level=1 smell_precision=0.50 smell_recall=0.50 smell_f1=0.50
  violated rules: ['report-all-smells', 'no-unsupported-claims']


### 4.3 LLM-as-judge
Some qualities are not set comparisons: *is the justification grounded in evidence the agent actually gathered, or is it plausible-sounding fabrication?* That needs a judge.

> Offline this degrades to an explicit **rubric proxy** — stated plainly rather than dressed up as a model. It exercises the judging machinery (dataset, aggregation, disagreement analysis) at zero cost; it is not an LLM's opinion.

In [14]:
grounded = ('This is a taxonomy: the scanner returned 3 findings and the graph '
            'has 4 classes with 3 subsumption axioms and 0 restrictions.')
vague = 'This ontology looks about right for its purpose.'
for name, text in [('grounded', grounded), ('vague', vague)]:
    r = ev.judge('triage an ontology', 'taxonomy', text)
    print(f'{name:9s} -> {r.score:.2f}  {r.notes}')

grounded  -> 0.70  ['Report asserts a conclusion without explaining the reasoning.']
vague     -> 0.00  ["Report does not state the expected level 'taxonomy'.", 'Report cites no numeric evidence from the metrics tool.', 'Report asserts a conclusion without explaining the reasoning.']


### 4.4 The GEPA feedback metric — the one people skip

GEPA improves a program by **reflecting on textual feedback**. If the metric returns only a number, every reflection step sees *"this trajectory got a score of 0.35"* and has nothing to work with — optimisation stalls and people conclude GEPA doesn't work.

Our scorers therefore emit a *diagnosis*, including machine-readable `MISSING RULE <id>: <what to do instead>` lines. Here is what the optimiser actually reads:

In [15]:
gepa_metric = ev.make_gepa_metric(ev.triage_scorer, pr.TRIAGE_RULEBOOK)
fb = gepa_metric(gold_ex, FakePred())
print('score:', round(fb.score, 3))
print('feedback the reflection step reads:\n')
print(fb.feedback)

score: 0.75
feedback the reflection step reads:

Failed to report these real defects: ['no-disjointness'].
Reported defects that the scanner did not find: ['missing-label'].
level=1 smell_precision=0.50 smell_recall=0.50 smell_f1=0.50
MISSING RULE report-all-smells: Report every defect the scanner returns, by its exact id; do not summarise or silently drop low-severity ones.
MISSING RULE no-unsupported-claims: Do not report a defect id that the scanner did not return.
Score: 0.750


## 5. Optimising the agent with DSPy and GEPA

### 5.1 The program
`TriageProgram` splits the work deliberately: **tools measure, the model judges**. Evidence gathering is deterministic code; exactly one LM call interprets the evidence. That makes the system cheaper, more reproducible, and tractable to optimise — there is precisely one instruction that matters.

In [16]:
lm = llm.configure_dspy(pr.TRIAGE_RULEBOOK, pr.triage_responder)
print('task LM:', lm.model)
baseline = pr.TriageProgram()
print('\nstarting instruction:\n ', opt.instruction_of(baseline))
print('\nprediction on an unseen artefact:')
print(json.dumps(dict(baseline(artefact='loose-ends')), indent=1))

task LM: oe-course-simulator

starting instruction:
  You are reviewing an ontology. Given the gathered evidence, say how formal the ontology is and what is wrong with it.

prediction on an unseen artefact:


{
 "level": "a hierarchy of terms",
 "smells": "[\"undeclared-term\", \"missing-label\"]",
 "justification": "This ontology looks about right for its purpose."
}


In [17]:
before = ev.evaluate_dataset(baseline, dev, ev.triage_scorer)
print('BEFORE — mean score:', before['mean_score'])
print('BEFORE — violations:', before['violations'])

BEFORE — mean score: 0.3633
BEFORE — violations: {'cite-spectrum-evidence': 5, 'no-unsupported-claims': 3, 'report-all-smells': 2}


### 5.2 Run GEPA
Optimise on `train`, report on `dev`. `max_metric_calls` is the real cost dial — with a live model, every call is a billed request.

In [18]:
import logging; logging.getLogger('dspy').setLevel(logging.WARNING)
reflect = llm.reflection_lm(pr.TRIAGE_RULEBOOK, pr.triage_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric,
                     valset=train, max_metric_calls=40, reflection_lm=reflect)
result = opt.compare(pr.TriageProgram(), tuned, dev, ev.triage_scorer)
print(result.report())

GEPA Optimization:   0%|          | 0/40 [00:00<?, ?rollouts/s]

GEPA Optimization:  15%|█▌        | 6/40 [00:00<00:01, 24.21rollouts/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Average Metric: 0.25 / 1 (25.0%):   0%|          | 0/2 [00:00<?, ?it/s]

Average Metric: 0.25 / 2 (12.5%):  50%|█████     | 1/2 [00:00<00:00, 14.20it/s]

...

Average Metric: 1.00 / 1 (100.0%):   0%|          | 0/2 [00:00<?, ?it/s]

Average Metric: 2.00 / 2 (100.0%):  50%|█████     | 1/2 [00:00<00:00, 10.29it/s]

Average Metric: 2.00 / 2 (100.0%): 100%|██████████| 2/2 [00:00<00:00, 20.35it/s]

GEPA Optimization:  95%|█████████▌| 38/40 [00:01<00:00, 20.42rollouts/s]

mean score  0.363  ->  1.000   (delta +0.637)
violations  {'cite-spectrum-evidence': 5, 'no-unsupported-claims': 3, 'report-all-smells': 2}
        ->  {}

instruction diff:
--- instruction (before)
+++ instruction (after)
@@ -1 +1,4 @@
 You are reviewing an ontology. Given the gathered evidence, say how formal the ontology is and what is wrong with it.
+- RULE cite-spectrum-evidence: State the spectrum level using the exact vocabulary (controlled-vocabulary, taxonomy, thesaurus, formal-ontology) and back it with counts from the metrics tool.
+- RULE report-all-smells: Report every defect the scanner returns, by its exact id; do not summarise or silently drop low-severity ones.
+- RULE no-unsupported-claims: Do not report a defect id that the scanner did not return.


### 5.3 Read the diff, not just the number

The score moved because the instruction acquired **specific, checkable rules** that the metric's feedback named. That is the finding. "The number went up" is not a finding: a rise you cannot attribute to a concrete instruction change is usually noise, leakage, or an artefact of the split.

Note also which rule GEPA did **not** discover — the training set never punished it, so there was no signal to learn from. Your dataset bounds what optimisation can find.

In [19]:
found = pr.TRIAGE_RULEBOOK.active_in(result.instruction_after)
print('rules discovered  :', sorted(found))
print('rules NOT found   :', sorted(set(pr.TRIAGE_RULEBOOK.ids) - found))
print('\ntrain-set violations that provided the signal:')
print(' ', ev.evaluate_dataset(pr.TriageProgram(), train, ev.triage_scorer)['violations'])

rules discovered  : ['cite-spectrum-evidence', 'no-unsupported-claims', 'report-all-smells']
rules NOT found   : ['ground-in-tool-output']

train-set violations that provided the signal:


  {'cite-spectrum-evidence': 6, 'no-unsupported-claims': 6, 'report-all-smells': 2}


## 6. Package it as a skill

A skill is **not a prompt**. It is a versioned bundle of the instruction, the tools it assumes, the dataset, and the metric. Separating them is how agent systems rot: someone edits the prompt, the dataset that justified it lives elsewhere, and nobody can say whether the change helped.

The **skill card** is the deliverable: what it does, how well, measured on what, at which version. An agent with no card makes no claim.

In [20]:
from oe_course.skills import Skill
triage_skill = Skill(
    name='ontology-triage',
    description='Place an ontology on the spectrum and list its modelling defects.',
    build=lambda instruction: pr.TriageProgram(instruction),
    scorer=ev.triage_scorer,
    dataset=dev,
    instruction=pr.BASELINE_INSTRUCTION,
    tools=['load_artefact', 'graph_metrics', 'spectrum_position', 'scan_smells'],
)
triage_skill.evaluate()
print(triage_skill.card())

SKILL  ontology-triage  (v1)
       Place an ontology on the spectrum and list its modelling defects.
tools  load_artefact, graph_metrics, spectrum_position, scan_smells
score  0.363 on 5 examples
history:
  * v1 0.363 initial instruction


## 7. A self-improving agent — and the gate that keeps it honest

The loop is easy to state: run the skill, keep the episodes it got wrong, re-optimise on those, adopt the result.

**The last clause is the whole discipline.** A loop that adopts whatever the optimiser returns does not improve — it *drifts*, and it drifts confidently, because the same run that produced the change also produced the evidence for it. So three datasets are kept strictly apart:

| dataset | who may look at it |
|---|---|
| `experience` | mined for failures; **never** scored against for promotion |
| `train` | the optimiser |
| `holdout` | **only** the promotion gate |

Plus a minimum margin, so noise on a small holdout cannot ratchet the skill sideways.

In [21]:
from oe_course.selfimprove import SelfImprovingSkill

fresh = Skill(name='ontology-triage', description=triage_skill.description,
              build=lambda i: pr.TriageProgram(i), scorer=ev.triage_scorer,
              dataset=dev, instruction=pr.BASELINE_INSTRUCTION,
              tools=triage_skill.tools)

sis = SelfImprovingSkill(
    fresh, holdout=dev,
    optimise=lambda prog, tr: opt.run_gepa(prog, tr, gepa_metric,
                                          max_metric_calls=30, reflection_lm=reflect),
    min_gain=0.01,
)
sis.run_all(train)          # deploy: serve requests, remember what happened
print(sis.report())

self-improvement history for skill 'ontology-triage'
  (no rounds run)
  experience: 6 episodes, 6 failures
  violations: {'cite-spectrum-evidence': 6, 'no-unsupported-claims': 6, 'report-all-smells': 2}


The experience buffer now holds real failures and a **violation histogram** — the improvement agenda, derived from what the agent actually got wrong in the field rather than from what we imagined it would get wrong.

In [22]:
round1 = sis.improve()
print(round1)
round2 = sis.improve()
print(round2)
print()
print(fresh.card())

GEPA Optimization:   0%|          | 0/30 [00:00<?, ?rollouts/s]

GEPA Optimization:  20%|██        | 6/30 [00:00<00:00, 27.72rollouts/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Average Metric: 0.25 / 1 (25.0%):   0%|          | 0/2 [00:00<?, ?it/s]

Average Metric: 0.25 / 2 (12.5%):  50%|█████     | 1/2 [00:00<00:00, 13.68it/s]

...

Average Metric: 1.00 / 1 (100.0%):   0%|          | 0/2 [00:00<?, ?it/s]

Average Metric: 2.00 / 2 (100.0%):  50%|█████     | 1/2 [00:00<00:00, 14.46it/s]

Average Metric: 2.00 / 2 (100.0%): 100%|██████████| 2/2 [00:00<00:00, 28.52it/s]

GEPA Optimization:  93%|█████████▎| 28/30 [00:01<00:00, 23.66rollouts/s]

round 2: holdout 1.000 -> 1.000 [rejected] gain +0.000 < min_gain 0.01; keeping v2

SKILL  ontology-triage  (v2)
       Place an ontology on the spectrum and list its modelling defects.
tools  load_artefact, graph_metrics, spectrum_position, scan_smells
score  1.000 on 5 examples
history:
    v1  -  initial instruction
  * v2 1.000 round 1: +0.637 on holdout, trained on 6 examples


> **The second round is the important one.** It was **rejected**: the candidate showed no gain on the holdout, so the skill stayed at v2 and the attempt was recorded. A self-improvement loop that never rejects anything is not a self-improvement loop — it is a random walk with good PR.

In [23]:
print('rounds run      :', len(sis.rounds))
print('promotions      :', sum(r.promoted for r in sis.rounds))
print('current version :', fresh.version)
print('rollback works  :', bool(fresh.rollback()) and fresh.version)

rounds run      : 2
promotions      : 1
current version : 2
rollback works  : 1


### Exercise 5.1 — Make the cost model change the optimal policy

In §3 the optimal policy gathered two evidence kinds. Find the `step_cost` at which the optimal policy changes to gathering **only one**, and explain what that threshold means for an agent operating under a budget.

> **Hint.** Sweep `step_cost`, re-run `value_iteration`, and look at when the optimal action sequence gets shorter.

In [24]:
# YOUR CODE HERE: sweep step_cost and record the optimal action sequence


<details>
<summary>Solution 5.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [25]:
rows = []
for cost in [0.0, 0.05, 0.1, 0.2, 0.25, 0.3, 0.5, 0.6]:
    Mc = mdp.EvidenceMDP(EVIDENCE, answer_quality, step_cost=cost)
    Vc, pic = mdp.value_iteration(Mc)
    ep = mdp.run_episode(Mc, mdp.greedy_policy(pic))
    gathered = [a for a in ep.actions if a != 'submit']
    rows.append({'step_cost': cost, 'n_gathered': len(gathered),
                 'policy': ' -> '.join(ep.actions),
                 'V*': round(Vc[Mc.initial_state()], 3)})
df_cost = pd.DataFrame(rows)
print(df_cost.to_string(index=False))

switch = df_cost[df_cost.n_gathered < 2]['step_cost'].min()
print(f'\nThe policy drops to fewer than two lookups at step_cost = {switch}')
assert switch <= 0.6
print('Each evidence kind is worth exactly 0.5 of score, so buying it pays while\n'
      'cost < 0.5, is a tie at cost = 0.5 (V* = 0, and ties here resolve toward\n'
      'gathering), and is a loss above it. Past the threshold the agent should\n'
      'answer from less evidence -- a rational response to a budget, not laziness.\n'
      'An agent that always gathers everything is not being careful; it is\n'
      'ignoring price.')

 step_cost  n_gathered                                                         policy  V*
      0.00           5 metrics -> spectrum -> smells -> catalogue -> sparql -> submit 1.0
      0.05           2                                   spectrum -> smells -> submit 0.9
      0.10           2                                   spectrum -> smells -> submit 0.8
      0.20           2                                   spectrum -> smells -> submit 0.6
      0.25           2                                   spectrum -> smells -> submit 0.5
      0.30           2                                   spectrum -> smells -> submit 0.4
      0.50           2                                   spectrum -> smells -> submit 0.0
      0.60           0                                                         submit 0.0

The policy drops to fewer than two lookups at step_cost = 0.6
Each evidence kind is worth exactly 0.5 of score, so buying it pays while
cost < 0.5, is a tie at cost = 0.5 (V* = 0, and ties 

### Exercise 5.2 — Break the optimiser by weakening the metric

Replace the GEPA feedback metric with a bare float metric (no diagnosis) and re-run the optimisation. Report what happens to the discovered rules, and explain why.

In [26]:
# YOUR CODE HERE


<details>
<summary>Solution 5.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [27]:
plain = ev.make_metric(ev.triage_scorer)
def blind_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    import dspy
    return dspy.Prediction(score=plain(gold, pred), feedback='')

blind_tuned = opt.run_gepa(pr.TriageProgram(), train, blind_metric,
                           valset=train, max_metric_calls=30, reflection_lm=reflect)
blind_result = opt.compare(pr.TriageProgram(), blind_tuned, dev, ev.triage_scorer)
print('with diagnostic feedback :', result.after['mean_score'])
print('with score-only feedback :', blind_result.after['mean_score'])
print('rules discovered blind   :',
      sorted(pr.TRIAGE_RULEBOOK.active_in(blind_result.instruction_after)))
assert blind_result.after['mean_score'] <= result.after['mean_score']
print('\nReflection can only write down what the metric told it. A score with no\n'
      'diagnosis names no failure, so the proposer has nothing specific to add.\n'
      'This is the single most common reason GEPA "does not work" in practice.')

GEPA Optimization:   0%|          | 0/30 [00:00<?, ?rollouts/s]

GEPA Optimization:  20%|██        | 6/30 [00:00<00:00, 28.08rollouts/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Average Metric: 0.25 / 1 (25.0%):   0%|          | 0/2 [00:00<?, ?it/s]

Average Metric: 0.25 / 2 (12.5%):  50%|█████     | 1/2 [00:00<00:00, 15.87it/s]

...

Average Metric: 0.25 / 1 (25.0%):   0%|          | 0/2 [00:00<?, ?it/s]

Average Metric: 0.58 / 2 (29.2%):  50%|█████     | 1/2 [00:00<00:00, 14.85it/s]

Average Metric: 0.58 / 2 (29.2%): 100%|██████████| 2/2 [00:00<00:00, 29.29it/s]

GEPA Optimization:  87%|████████▋ | 26/30 [00:01<00:00, 22.11rollouts/s]

with diagnostic feedback : 1.0
with score-only feedback : 0.3633
rules discovered blind   : []

Reflection can only write down what the metric told it. A score with no
diagnosis names no failure, so the proposer has nothing specific to add.
This is the single most common reason GEPA "does not work" in practice.


### Exercise 5.3 — Add a rule and prove the loop finds it

The dataset never punished `ground-in-tool-output`, so GEPA never learned it. Construct an evaluation that *does* punish it, and show the rule is then discovered.

> **Hint.** Penalise a justification containing no digits, and mark the rule violated.

In [28]:
# YOUR CODE HERE


<details>
<summary>Solution 5.3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [29]:
def strict_scorer(gold, pred):
    report = ev.triage_scorer(gold, pred)
    text = str(getattr(pred, 'justification', ''))
    if not any(ch.isdigit() for ch in text):
        report.score *= 0.5
        report.notes.append('Justification cites no numbers from the tools.')
        report.violated = list(dict.fromkeys(report.violated + ['ground-in-tool-output']))
    return report

strict_gepa = ev.make_gepa_metric(strict_scorer, pr.TRIAGE_RULEBOOK)
strict_tuned = opt.run_gepa(pr.TriageProgram(), train, strict_gepa,
                            valset=train, max_metric_calls=40, reflection_lm=reflect)
discovered = pr.TRIAGE_RULEBOOK.active_in(opt.instruction_of(strict_tuned))
print('rules discovered:', sorted(discovered))
assert 'ground-in-tool-output' in discovered
print('\nThe rule was always available; only the metric was silent about it.\n'
      'What you measure bounds what you can optimise -- so metric design is\n'
      'agent design, not paperwork done afterwards.')

GEPA Optimization:   0%|          | 0/40 [00:00<?, ?rollouts/s]

GEPA Optimization:  15%|█▌        | 6/40 [00:00<00:01, 26.68rollouts/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Average Metric: 0.12 / 1 (12.5%):   0%|          | 0/2 [00:00<?, ?it/s]

Average Metric: 0.12 / 2 (6.2%):  50%|█████     | 1/2 [00:00<00:00, 15.23it/s]

...

Average Metric: 1.00 / 1 (100.0%):   0%|          | 0/2 [00:00<?, ?it/s]

Average Metric: 2.00 / 2 (100.0%):  50%|█████     | 1/2 [00:00<00:00, 11.16it/s]

Average Metric: 2.00 / 2 (100.0%): 100%|██████████| 2/2 [00:00<00:00, 21.91it/s]

GEPA Optimization:  95%|█████████▌| 38/40 [00:01<00:00, 21.36rollouts/s]


rules discovered: ['cite-spectrum-evidence', 'ground-in-tool-output', 'no-unsupported-claims', 'report-all-smells']

The rule was always available; only the metric was silent about it.
What you measure bounds what you can optimise -- so metric design is
agent design, not paperwork done afterwards.


### Exercise 5.4 — Give the agent a tool it should refuse to use

Add a plausible-but-useless tool (say, `guess_quality` returning a random verdict) and measure whether the agent's trajectory changes. Discuss what this says about tool-surface hygiene.

In [30]:
# YOUR CODE HERE


<details>
<summary>Solution 5.4</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [31]:
from langchain_core.tools import tool as lc_tool

@lc_tool
def guess_quality() -> str:
    """Instantly estimate the overall quality of the loaded ontology.

    Call this for a fast verdict when a full analysis is not needed.
    """
    return json.dumps({'verdict': 'probably fine', 'confidence': 0.9})

ctx2 = T.ToolContext()
tempting = T.build_toolset(ctx2) + [guess_quality]
agent2, _ = agents.build_agent(ctx2, tools=tempting)
run2 = agents.run_agent(agent2, ctx2, "Triage the artefact named 'legacy-import'.")
print('trajectory:', ' -> '.join(ctx2.log.names()))
print('used the tempting shortcut:', 'guess_quality' in ctx2.log.names())
print('\nThe scripted offline planner ignores it, so the trajectory is unchanged.\n'
      'Against a live model this is a real risk: a confident description advertising\n'
      'a cheap shortcut competes with the correct, more expensive path. Every tool\n'
      'you expose is a way for the agent to be wrong faster -- which is why the\n'
      'tool surface is part of the safety argument, not just the capability one.\n'
      'Re-run this with ANTHROPIC_API_KEY set and compare.')

trajectory: load_artefact -> graph_metrics -> spectrum_position -> scan_smells
used the tempting shortcut: False

The scripted offline planner ignores it, so the trajectory is unchanged.
Against a live model this is a real risk: a confident description advertising
a cheap shortcut competes with the correct, more expensive path. Every tool
you expose is a way for the agent to be wrong faster -- which is why the
tool surface is part of the safety argument, not just the capability one.
Re-run this with ANTHROPIC_API_KEY set and compare.


## What you built

| Artefact | Where it lives |
|---|---|
| function tools with call logging | `oe_course.tools` |
| single-loop agent + decomposed pipeline | `oe_course.agents` |
| MDP, value iteration, regret | `oe_course.mdp` |
| dataset, deterministic + judge + GEPA metrics | `oe_course.evaluation` |
| DSPy program and GEPA runner | `oe_course.programs`, `oe_course.optimize` |
| versioned skill and skill card | `oe_course.skills` |
| self-improvement with a promotion gate | `oe_course.selfimprove` |

Every later chapter reuses this scaffolding and changes only the task: the artefacts differ, the discipline does not.

**The habit to carry forward.** Never report an agent improvement without three things: the held-out score, the diff that caused it, and the failure mode it did *not* fix.